# Multimodel integration (Groq, Google)

In [ ]:
import torch

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"config\.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GOOGLE_GENAI_API_KEY = os.getenv("GOOGLE_GENAI_API_KEY")


# --- ROCm/CUDA device check ---
# ROCm exposes itself to PyTorch through the same torch.cuda API as NVIDIA CUDA,
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU detected: {device_name} ({total_vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected by torch — falling back to CPU. Check your ROCm/torch install.")


GPU detected: AMD Radeon RX 7900 XT (20.0 GB VRAM)


## Groq integration

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
        "openai/gpt-oss-20b",
        model_provider="groq",
        api_key=GROQ_API_KEY,
    )

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022D3FE273E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022D3FE27B00>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [28]:
response = model.invoke("Hello, how are you?")
response

AIMessage(content='Hello! I’m doing great—thanks for asking. How can I help you today?', additional_kwargs={'reasoning_content': 'The user says: "Hello, how are you?" They likely want a friendly response. The user hasn\'t asked a specific question, so we can respond with a friendly greeting and ask them what they need. According to policies, we can respond. There\'s no disallowed content. So produce a friendly response.'}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 77, 'total_tokens': 166, 'completion_time': 0.124641704, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.003685799, 'prompt_tokens_details': None, 'queue_time': 0.016552026, 'total_time': 0.128327503}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a021a7-92a2-7ba1-95f2-37539316a46c-0', tool_calls=[], invalid_tool_calls=[], usage

In [29]:
response.content

'Hello! I’m doing great—thanks for asking. How can I help you today?'

## Gemini integration

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model( # Method 1: Using init_chat_model with the model string
    "google_genai:gemini-3.5-flash-lite",
    api_key=GOOGLE_GENAI_API_KEY,
    )

response = model.invoke("Why do parrots have such colorful feathers?")
response.content

[{'type': 'text',
  'text': 'Parrots are among the most brilliantly colored birds in the world, and their vibrant feathers aren\'t just for show. Evolutionarily speaking, these striking colors serve several crucial survival and social functions. \n\nHere are the main reasons why parrots have such colorful feathers:\n\n### 1. Camouflage in the Rainforest Canopy\nWhile bright red, green, and blue might look like they would stand out, in a tropical rainforest, they actually serve as camouflage. \n* **The Green Paradox:** Most parrots are predominantly green. In the dappled sunlight of a lush jungle canopy, green feathers act as perfect camouflage against leaves and vines, hiding them from predators like hawks and eagles.\n* **Breaking Up the Outline:** Other bright colors (like the red and yellow on a Macaw) actually break up the bird’s silhouette when they are sitting among brightly colored tropical flowers, fruits, and sun-dappled shadows. This is called *disruptive coloration*.\n\n### 

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI( # Method 2: Using the ChatGoogleGenerativeAI class directly
    model="gemini-3.5-flash-lite",
    google_api_key=GOOGLE_GENAI_API_KEY
)

response = model.invoke("Why do parrots have such colorful feathers?")
response.content

[{'type': 'text',
  'text': "Parrots are among the most brilliantly colored birds in the world, featuring vibrant greens, reds, yellows, and blues. Nature didn't paint them this way just to look pretty; these striking colors serve several crucial survival and social functions. \n\nHere are the main reasons why parrots have such colorful feathers:\n\n### 1. Camouflage in the Rainforest\nWhile bright neon colors might sound like they would make a parrot stand out, in their native tropical rainforest habitats, they actually provide great camouflage. \n* **The Green Paradox:** Many parrots are predominantly green. In the lush, dappled sunlight of a rainforest canopy, a green parrot is almost entirely invisible to predators like hawks and eagles.\n* **Breaking the Outline:** Other bright colors (like reds, yellows, and blues) often mimic tropical flowers, fruits, or the play of bright sunlight and deep shadows in the jungle canopy, breaking up the bird's silhouette so predators overlook the